In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from blackjack.dealer_sim import _load_and_precompute_combo_data

In [2]:
precomputed_combinations_with_counts_no_bj = _load_and_precompute_combo_data(
    "../combinations/combinations_with_counts_no_bj_d13.pkl"
)
precomputed_combinations_with_counts = _load_and_precompute_combo_data(
    "../combinations/combinations_with_counts_d13.pkl"
)

In [3]:
type(precomputed_combinations_with_counts_no_bj)

dict

In [4]:
precomputed_combinations_with_counts_no_bj.keys()

dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16])

In [5]:
type(precomputed_combinations_with_counts_no_bj[1])

dict

In [6]:
precomputed_combinations_with_counts_no_bj[1].keys()

dict_keys([2, 3, 4, 5, 6, 7, 8, 9, 10, 11])

In [7]:
precomputed_combinations_with_counts_no_bj[1][2]

{'stand_combos_data': [],
 'other_combos_data': [((2,), 1, 2, 4, False),
  ((3,), 1, 3, 5, False),
  ((4,), 1, 4, 6, False),
  ((5,), 1, 5, 7, False),
  ((6,), 1, 6, 8, False),
  ((7,), 1, 7, 9, False),
  ((8,), 1, 8, 10, False),
  ((9,), 1, 9, 11, False),
  ((10,), 1, 10, 12, False),
  ((11,), 1, 11, 13, True)],
 'bust_combos_data': [],
 'stand_values': array([], dtype=float64)}

In [8]:
# Compare nested dicts with readable diffs
from collections.abc import Mapping

def compare_nested(a, b, path=(), diffs=None, max_diffs=50):
    if diffs is None:
        diffs = []
    if len(diffs) >= max_diffs:
        return diffs

    num_types = (int, float, np.integer, np.floating)
    if type(a) != type(b) and not (isinstance(a, num_types) and isinstance(b, num_types)):
        diffs.append((path, f"type mismatch {type(a).__name__} vs {type(b).__name__}"))
        return diffs

    if isinstance(a, Mapping):
        a_keys = set(a.keys())
        b_keys = set(b.keys())
        for k in sorted(a_keys - b_keys):
            diffs.append((path + (k,), "missing in B"))
            if len(diffs) >= max_diffs:
                return diffs
        for k in sorted(b_keys - a_keys):
            diffs.append((path + (k,), "missing in A"))
            if len(diffs) >= max_diffs:
                return diffs
        for k in sorted(a_keys & b_keys):
            compare_nested(a[k], b[k], path + (k,), diffs, max_diffs)
            if len(diffs) >= max_diffs:
                return diffs
        return diffs

    if isinstance(a, np.ndarray):
        if a.shape != b.shape or not np.array_equal(a, b):
            diffs.append((path, f"ndarray mismatch shape {a.shape} vs {b.shape}"))
        return diffs

    if isinstance(a, (list, tuple)):
        if len(a) != len(b):
            diffs.append((path, f"length mismatch {len(a)} vs {len(b)}"))
            if len(diffs) >= max_diffs:
                return diffs
        for i, (ai, bi) in enumerate(zip(a, b)):
            compare_nested(ai, bi, path + (i,), diffs, max_diffs)
            if len(diffs) >= max_diffs:
                return diffs
        return diffs

    if isinstance(a, num_types) and isinstance(b, num_types):
        if not np.isclose(a, b, equal_nan=True):
            diffs.append((path, f"value mismatch {a} vs {b}"))
        return diffs

    if a != b:
        diffs.append((path, f"value mismatch {a} vs {b}"))
    return diffs

def print_diffs(diffs, max_preview=20):
    print(f"diffs: {len(diffs)}")
    for path, msg in diffs[:max_preview]:
        pretty_path = " -> ".join(str(p) for p in path) or "root"
        print(f"{pretty_path}: {msg}")


In [9]:
max_diffs = 50
diffs = compare_nested(
    precomputed_combinations_with_counts_no_bj,
    precomputed_combinations_with_counts,
    max_diffs=max_diffs,
)
print_diffs(diffs)


diffs: 0


In [10]:
# Sanity check: verify the comparison function catches real differences
test_a = {"x": 1, "y": [1, 2, 3]}
test_b = {"x": 2, "y": [1, 2, 4]}
test_diffs = compare_nested(test_a, test_b)
print("Sanity check (should show 2 diffs):")
print_diffs(test_diffs)

# Also verify dict identity vs equality
print(f"\nAre they the exact same object? {precomputed_combinations_with_counts_no_bj is precomputed_combinations_with_counts}")
print(f"Are they equal? {precomputed_combinations_with_counts_no_bj == precomputed_combinations_with_counts}")


Sanity check (should show 2 diffs):
diffs: 2
x: value mismatch 1 vs 2
y -> 2: value mismatch 3 vs 4

Are they the exact same object? False


ValueError: The truth value of an empty array is ambiguous. Use `array.size > 0` to check that an array is not empty.

In [11]:
# Check if the source pickle files are byte-identical
from pathlib import Path
import hashlib

combos_dir = Path("../combinations")
file1 = combos_dir / "combinations_with_counts_no_bj_d13.pkl"
file2 = combos_dir / "combinations_with_counts_d13.pkl"

hash1 = hashlib.md5(file1.read_bytes()).hexdigest()
hash2 = hashlib.md5(file2.read_bytes()).hexdigest()

print(f"File 1 ({file1.name}): {hash1}")
print(f"File 2 ({file2.name}): {hash2}")
print(f"Files are byte-identical: {hash1 == hash2}")


File 1 (combinations_with_counts_no_bj_d13.pkl): 7c2231ff243cc2cd6cfc2948752b81bd
File 2 (combinations_with_counts_d13.pkl): 7c2231ff243cc2cd6cfc2948752b81bd
Files are byte-identical: True


In [12]:
# Compare JSON files between bj_not_checked and dealer_checked_no_bj
import json

dir_a = Path("../combinations/bj_not_checked")
dir_b = Path("../combinations/dealer_checked_no_bj")

files_a = sorted(dir_a.glob("*.json"))
files_b = sorted(dir_b.glob("*.json"))

print(f"Files in bj_not_checked: {len(files_a)}")
print(f"Files in dealer_checked_no_bj: {len(files_b)}")

# Check file names match
names_a = {f.name for f in files_a}
names_b = {f.name for f in files_b}
only_in_a = names_a - names_b
only_in_b = names_b - names_a

if only_in_a:
    print(f"Only in bj_not_checked: {only_in_a}")
if only_in_b:
    print(f"Only in dealer_checked_no_bj: {only_in_b}")
if not only_in_a and not only_in_b:
    print("Both directories have the same file names ✓")


Files in bj_not_checked: 110
Files in dealer_checked_no_bj: 110
Both directories have the same file names ✓


In [13]:
# Compare content of each JSON file pair
common_files = sorted(names_a & names_b)
files_with_diffs = []
total_diffs = 0

for fname in common_files:
    file_a = dir_a / fname
    file_b = dir_b / fname
    
    with open(file_a) as f:
        data_a = json.load(f)
    with open(file_b) as f:
        data_b = json.load(f)
    
    diffs = compare_nested(data_a, data_b, max_diffs=10)
    if diffs:
        files_with_diffs.append((fname, diffs))
        total_diffs += len(diffs)

print(f"Compared {len(common_files)} file pairs")
print(f"Files with differences: {len(files_with_diffs)}")
print(f"Total differences found: {total_diffs}")

if files_with_diffs:
    print("\n--- Files with differences ---")
    for fname, diffs in files_with_diffs[:10]:  # Show first 10 files
        print(f"\n{fname}:")
        print_diffs(diffs, max_preview=5)
else:
    print("\nAll JSON files are identical ✓")


Compared 110 file pairs
Files with differences: 0
Total differences found: 0

All JSON files are identical ✓


In [14]:
# Quick byte-level comparison of JSON files
byte_identical = []
byte_different = []

for fname in common_files:
    hash_a = hashlib.md5((dir_a / fname).read_bytes()).hexdigest()
    hash_b = hashlib.md5((dir_b / fname).read_bytes()).hexdigest()
    
    if hash_a == hash_b:
        byte_identical.append(fname)
    else:
        byte_different.append(fname)

print(f"Byte-identical files: {len(byte_identical)}")
print(f"Byte-different files: {len(byte_different)}")

if byte_different:
    print(f"\nFiles that differ: {byte_different[:20]}")  # Show first 20


Byte-identical files: 110
Byte-different files: 0
